# Read-only experiment progress

**Purpose:** show the six declared training runs and controller state without starting training. **Inputs:** frozen v3 protocol, pipeline and training JSON receipts. **Outputs:** `reports/ahr_jepa_v3/progress.json` and `.png`. **Runtime:** under one minute. **Resume:** fresh kernel, Run All; scientific checkpoints are read through their public receipts only. Missing reports stay missing. A recorded game may have an UNKNOWN outcome; training progress is not a strength metric.

In [ ]:
from pathlib import Path
import nbformat
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run within AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
        exec(compile(cell.source,'v3-shared:export','exec'),globals())


In [ ]:
import io
from datetime import datetime, timezone
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

def v3_refresh_progress(project):
    project=Path(project);directory=project/'reports/ahr_jepa_v3'
    directory.mkdir(parents=True,exist_ok=True)
    sources={};missing=[]
    def read(relative):
        p=project/relative
        if not p.exists():missing.append(relative);return None
        raw=p.read_bytes();value=json.loads(raw)
        sources[relative]={'sha256':hashlib.sha256(raw).hexdigest(),'bytes':len(raw)}
        return value
    config=read('config/ahr_jepa_v3.json')
    pipeline=read('reports/pipeline-ahr-jepa-v3.json')
    rows=[]
    if config:
        for index,seed in enumerate(config['training_seeds'],1):
            for arm in ('B','C'):
                run=f'ahr-v3-{arm}-s{index:02d}'
                report=read(f'reports/ahr_jepa_v3/{run}_training.json')
                row={'run_id':run,'arm':arm,'seed':seed,'status':'NO_REPORT','games':None,'optimizer_steps':None,
                     'planned_games':config['training']['games'],'planned_updates':config['training']['warmup_updates']+config['training']['games']*config['training']['updates_per_game']}
                if report:
                    row.update(status=report.get('status',report.get('phase','UNKNOWN')),
                        games=report.get('games_finished',report.get('completed_games')),
                        optimizer_steps=report.get('optimizer_steps',report.get('optimizer_step')),
                        unknown_games=report.get('unknown_games'))
                rows.append(row)
    snapshot={'schema':'ahr_v3_progress','generated_at_utc':datetime.now(timezone.utc).isoformat(),
        'pipeline':None if pipeline is None else {k:pipeline.get(k) for k in ('status','stage_index','current','updated_at_utc')},
        'training':rows,'source_inventory':sources,'missing_inputs':missing,
        'interpretation':'Progress only. Training completion and prediction loss do not establish playing strength.'}
    fig,axes=plt.subplots(1,2,figsize=(13,5.7),layout='constrained')
    fig.suptitle('AHR + predictive learning · three-seed experiment',fontsize=18,fontweight='bold',color='#243440')
    colors={'B':'#2b6787','C':'#b96b2c'}
    for ax,metric,planned,title in [(axes[0],'games','planned_games','Recorded self-play games'),(axes[1],'optimizer_steps','planned_updates','Optimizer updates')]:
        if rows:
            ys=np.arange(len(rows));labels=[f"{r['arm']} · seed {i//2+1}" for i,r in enumerate(rows)]
            for y,row in zip(ys,rows):
                ax.barh(y,row[planned],color='#e5e9ec',height=.58)
                value=row[metric]
                if value is not None:ax.barh(y,value,color=colors[row['arm']],height=.58)
                ax.text(row[planned]*1.02,y,'No report' if value is None else f'{value} / {row[planned]}',va='center',fontsize=10)
            ax.set_yticks(ys,labels);ax.invert_yaxis();ax.set_xlim(0,max(r[planned] for r in rows)*1.4)
        else:
            ax.text(.5,.5,'Protocol not frozen yet',transform=ax.transAxes,ha='center',va='center');ax.set_yticks([])
        ax.set_title(title,loc='left',fontsize=13,fontweight='bold');ax.set_xlabel('Count');ax.set_axisbelow(True);ax.grid(axis='x',color='#e5e9ec')
        for side in ('top','right'):ax.spines[side].set_visible(False)
    status='NOT_LAUNCHED' if pipeline is None else pipeline['status']
    current=None if pipeline is None else (pipeline.get('current') or {}).get('name')
    fig.supxlabel(f"{status} · {current or 'No active stage'}\nB: improved AHR · C: identical setup + latent prediction; UNKNOWN games keep separate outcome labels.",fontsize=10)
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=150);plt.close(fig)
    # Import only the original atomic write helpers in an isolated namespace.
    helpers={}
    for cell in nbformat.read(project/'notebooks/00_instance_setup_and_recovery.ipynb',4).cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'atomic:export','exec'),helpers)
    helpers['atomic_write_bytes'](directory/'progress.png',buffer.getvalue())
    helpers['atomic_json'](directory/'progress.json',snapshot)
    return snapshot


In [ ]:
snapshot=v3_refresh_progress(PROJECT)
print(json.dumps({'pipeline':snapshot['pipeline'],'runs':len(snapshot['training']),'missing_reports':len(snapshot['missing_inputs'])},indent=2))
from IPython.display import display,Image
display(Image(filename=str(PROJECT/'reports/ahr_jepa_v3/progress.png')))